# DeepExtractor Residuals — GravitySpy Classification (glitchgan pipeline, exact)

Runs DeepExtractor's extracted O3 residual glitches through **exactly** the same
GravitySpy classification pipeline as
`glitchgan/notebooks/gspy_classification.ipynb` — same whitening (pycbc, not gwpy),
same injection/centering math, same SNR rescaling, same `classify_signals` and
`plot_confusion` functions verbatim. Only the *source* of the glitches changes:
real DeepExtractor residuals (loaded from a `residuals.npz`) instead of
GlitchGAN-generated signals.

Purpose: `notebooks/gspy_classification_o3.ipynb` (gwpy-whitening, DeepExtractor's own
injection convention) gave near-zero accuracy with high-confidence wrong predictions
(Light_Modulation / Extremely_Loud regardless of true class). This notebook is a
controlled test — if the *exact* pipeline that GravitySpy is known to work with also
fails on these same residuals, the problem is the residuals themselves, not how they're
fed to GravitySpy.

## Dependencies

Same as `validate_gspy_o3.py` / `gspy_classification_o3.ipynb` — deepextractor + GravitySpy
runtime deps, GravitySpy itself installed with `--no-deps`, then patched:

```bash
pip install -e ".[gspy]"
pip install gravityspy==1.0.0 --no-deps
python patch_gspy.py
```

In [ ]:
%matplotlib inline

In [ ]:
import os

os.environ["CUDA_VISIBLE_DEVICES"] = ""
os.environ["TF_NUM_INTEROP_THREADS"] = "1"
os.environ["TF_NUM_INTRAOP_THREADS"] = "1"
os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "3")

import sys, io, shutil, warnings, logging
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from tqdm.notebook import tqdm
from IPython.display import Image as IPyImage, display as ipy_display

from deepextractor.utils.signal import whitened_snr_scaling
from deepextractor.utils.visualization import plot_q_transform

## Config

In [ ]:
# ── paths — adjust for your checkout ─────────────────────────────────────────
RESIDUALS_PATH = Path("/home/tom.dooney/deepextractor/evaluation/gspy_4s_o3_notebook/residuals.npz")
GSPY_MODEL     = Path("/home/tom.dooney/glitchgan/models/sidd-cqg-paper-O3-model.h5")
OUTPUT_DIR     = Path("/home/tom.dooney/deepextractor/evaluation/gspy_4s_o3_glitchgan_style")

os.makedirs(OUTPUT_DIR, exist_ok=True)

# ── glitch classes ────────────────────────────────────────────────────────────
LABEL_ORDER = [
    "Blip", "Fast_Scattering", "Koi_Fish",
    "Low_Frequency_Burst", "Scattered_Light", "Tomte", "Whistle",
]
NUM_CLASSES = len(LABEL_ORDER)

# ── GravitySpy noise / classification config — identical to gspy_classification.ipynb ──
IFO              = "H1"
SRATE            = 4096
GW_START, GW_END = 1262540000, 1262540040
CHANNEL          = f"{IFO}:GDS-CALIB_STRAIN"
INIT_TIME        = -20
EVENT_TIME       = 0
SNR_TARGET       = 50
NUM_CLASSIFY     = 3     # we only have 3 real residuals per class (vs GlitchGAN's unlimited samples)

print("Residuals exist:", RESIDUALS_PATH.exists())
print("Model exists   :", GSPY_MODEL.exists())

## Load DeepExtractor residuals

Replaces GlitchGAN's generator — `generated_signals`/`labels` end up with exactly the
shape/dtype `classify_signals` below expects, just sourced from real extracted
residuals instead of the GAN.

In [ ]:
data = np.load(RESIDUALS_PATH, allow_pickle=True)
generated_signals = data["residual"].astype(np.float32)   # (N, 16384) — DeepExtractor residuals
labels            = data["true_label"]                    # (N,) string labels

print(f"generated_signals: {generated_signals.shape}")
print(f"labels           : {labels.shape}")
for cls in LABEL_ORDER:
    print(f"  {cls:22s} {(labels == cls).sum()}")

## Q-scan each sample before classifying

Sanity check with the same `plot_q_transform` utility `validate_gspy_o3.py` uses, so we
can see whether each residual's morphology looks like the labeled glitch class *before*
trusting GravitySpy's verdict on it.

In [ ]:
for i, (sig, lbl) in enumerate(zip(generated_signals, labels)):
    fig, ax = plt.subplots(figsize=(8, 4))
    plot_q_transform(
        sig, srate=SRATE, whiten=False,
        qrange=[10, 10], frange=[10, 1200],
        ax=ax, colourbar=True,
    )
    ax.set_title(f"[{i+1}/{len(generated_signals)}] {lbl}", fontsize=11)
    plt.tight_layout()
    plt.show()

## GravitySpy Classification

In [ ]:
from gwpy.timeseries import TimeSeries
from gravityspy.classify import classify
import gravityspy.ml.labelling_test_glitches as _lgt

warnings.filterwarnings("ignore")
for _log in ["gravityspy", "gwpy", "astropy", "tensorflow"]:
    logging.getLogger(_log).setLevel(logging.ERROR)
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"

GSPY_PLOT_DIR = str(OUTPUT_DIR / "gspy_tmp")


def classify_signals(generated_signals, labels, white_noise, noise, label_order, tag):
    """Classify signals using GravitySpy.
    Injection follows evaluation2 exactly: pycbc copy → pycbc += → gwpy TimeSeries.
    """
    ifo          = IFO
    srate        = SRATE
    init_time    = INIT_TIME
    channel_name = CHANNEL
    path_to_model = str(GSPY_MODEL)
    snr_target   = SNR_TARGET

    shutil.rmtree(GSPY_PLOT_DIR, ignore_errors=True)
    os.makedirs(GSPY_PLOT_DIR, exist_ok=True)

    rows  = []
    total = NUM_CLASSIFY * len(label_order)
    with tqdm(total=total, desc=f"Classifying [{tag}]", unit="glitch") as pbar:
        for class_label in label_order:
            class_indices  = np.where(labels == class_label)[0]
            chosen_indices = np.random.choice(class_indices, NUM_CLASSIFY, replace=False)

            for idx in chosen_indices:
                glitch = generated_signals[idx].copy()
                glitch = whitened_snr_scaling(glitch, snr_target)

                len_glitch = len(glitch)
                length     = noise.shape[-1]
                t_inj      = 0.5 * length / srate
                id_start   = int((t_inj * srate / length) * len(white_noise)) - len_glitch // 2

                injected_noise = white_noise.copy()
                injected_noise[id_start:id_start + len_glitch] += glitch

                glitch_series = TimeSeries(
                    injected_noise, t0=init_time, sample_rate=srate, name=ifo
                )

                try:
                    result = classify(
                        event_time=EVENT_TIME,
                        channel_name=channel_name,
                        path_to_cnn=path_to_model,
                        timeseries=glitch_series,
                        plot_directory=GSPY_PLOT_DIR,
                    )
                    rows.append({
                        'true_label': class_label,
                        'pred_label': result['ml_label'].value[0],
                        'confidence': result['ml_confidence'].value[0],
                    })
                except Exception as e:
                    print(f'  ⚠ {class_label}[{idx}]: {type(e).__name__}: {e}')
                    rows.append({'true_label': class_label, 'pred_label': 'Error', 'confidence': 0.0})
                pbar.update(1)

    return pd.DataFrame(rows)

## Fetch and whiten clean noise

pycbc-based whitening, `remove_corrupted=False` — identical to the reference notebook,
**not** the gwpy-based whitening `gspy_classification_o3.ipynb`/`validate_gspy_o3.py` use.

In [ ]:
print("Fetching open data and whitening...")
noise = TimeSeries.fetch_open_data(IFO, GW_START, GW_END, sample_rate=SRATE)
noise = noise.to_pycbc()
white_noise, psd = noise.whiten(
    len(noise) / (2 * SRATE),
    len(noise) / (4 * SRATE),
    remove_corrupted=False,
    return_psd=True,
)
print(f"white_noise: {len(white_noise)} samples  dtype: {white_noise.dtype}")

## Classify

In [ ]:
df_results = classify_signals(
    generated_signals, labels, white_noise, noise,
    label_order=LABEL_ORDER,
    tag=f"DeepExtractor residuals (O3, SNR={SNR_TARGET})",
)
df_results.to_csv(OUTPUT_DIR / "gspy_results.csv", index=False)
print(f'Saved: {OUTPUT_DIR / "gspy_results.csv"}')

## Confusion matrix

In [ ]:
def plot_confusion(df, tag, save_name):
    if df is None or len(df) == 0:
        print(f'No results for {tag}'); return None
    df = df[df['pred_label'] != 'Error']
    if len(df) == 0:
        print(f'All errors for {tag}'); return None

    pred_all  = sorted(df['pred_label'].unique())
    for lbl in LABEL_ORDER:
        if lbl not in pred_all:
            pred_all.append(lbl)
    pred_cols = ([l for l in LABEL_ORDER if l in pred_all]
                 + [l for l in pred_all if l not in LABEL_ORDER])

    count_matrix = pd.DataFrame(0, index=LABEL_ORDER, columns=pred_cols)
    conf_accum   = {(t, p): [] for t in LABEL_ORDER for p in pred_cols}
    for t, p, c in zip(df['true_label'], df['pred_label'], df['confidence']):
        if t in LABEL_ORDER and p in pred_cols:
            count_matrix.loc[t, p] += 1
            conf_accum[(t, p)].append(c)

    annot = pd.DataFrame('', index=LABEL_ORDER, columns=pred_cols)
    for t in LABEL_ORDER:
        for p in pred_cols:
            n = count_matrix.loc[t, p]
            annot.loc[t, p] = "0" if n == 0 else f"{n}\n({np.mean(conf_accum[(t,p)]):.2f})"

    total = count_matrix.values.sum()
    acc   = np.trace(count_matrix.values) / total if total > 0 else 0.0

    fig_w = max(10, len(pred_cols) * 1.1)
    sns.set(style='whitegrid', font_scale=1.0)
    fig, ax = plt.subplots(figsize=(fig_w, 6))
    sns.heatmap(count_matrix, annot=annot, fmt='', cmap='Blues', cbar=True,
                linewidths=0.5, linecolor='gray',
                annot_kws={'size': 8, 'color': 'black'}, ax=ax)
    ax.set_xlabel('Predicted Label', fontsize=12)
    ax.set_ylabel('True Label', fontsize=12)
    ax.set_title(f"Gravity Spy — {tag}   (accuracy = {acc:.1%})", fontsize=13)
    plt.xticks(rotation=45, ha='right', fontsize=8)
    plt.yticks(rotation=0, fontsize=9)
    plt.tight_layout()
    fig.savefig(OUTPUT_DIR / f'{save_name}.pdf', bbox_inches='tight')
    buf = io.BytesIO()
    fig.savefig(buf, format='png', dpi=150, bbox_inches='tight')
    plt.close(fig)
    buf.seek(0)
    ipy_display(IPyImage(buf.read()))
    print(f'{tag} accuracy: {acc:.3f}')
    return acc


acc = plot_confusion(df_results, f"DeepExtractor residuals (O3, SNR={SNR_TARGET})", "gspy_cm")